# Archive: move the loaded files

Moves the source files that one run loaded into bronze from the `bronze.landing_zone` volume to the `bronze.archive`
volume, in the workspace's default catalog (both volumes are created by `src/setup`).

- Which files: the distinct `_file_name` of the run's rows in `bronze.raw_mandi_transactions`. A file that arrived after the load is not touched.
- Where to: `/Volumes/<catalog>/bronze/archive/<run_id>/<file name>`.
- The last task of `mandi_daily`, so a failure in silver or gold leaves the files in the landing zone.

Parameters (both required):
- `file_path`: the same path or pattern as in `src/load_bronze`; its folder is the landing zone.
- `run_id`: id of the run whose files are moved.

If a move fails, the reason is printed and the notebook still completes (it fails when `fail_on_error` is `true`).

In [ ]:
dbutils.widgets.text("file_path", "", "File path")
dbutils.widgets.text("run_id", "", "Run id")
dbutils.widgets.text("fail_on_error", "false", "Fail the job on error (true/false)")

In [ ]:
import os

from lib_mandi.extract import extract__bronze_file_names
from lib_mandi.load import load__archive_file
from lib_mandi.utility import get__default_catalog

file_path = dbutils.widgets.get("file_path").strip()
run_id = dbutils.widgets.get("run_id").strip()

if not file_path or not run_id:
    raise ValueError("Widgets 'file_path' and 'run_id' are required")

catalog = get__default_catalog(spark)
if not catalog:
    raise RuntimeError("Could not read the default catalog (see the error printed above)")

landing_dir = os.path.dirname(file_path)
archive_dir = f"/Volumes/{catalog}/bronze/archive"

file_names = extract__bronze_file_names(spark, run_id, f"{catalog}.bronze.raw_mandi_transactions")
archived = file_names is not None
for file_name in file_names or []:
    if not load__archive_file(os.path.join(landing_dir, file_name), archive_dir, run_id):
        archived = False

if not archived and dbutils.widgets.get("fail_on_error").strip().lower() == "true":
    raise RuntimeError("Archive did not finish (see the reason printed above)")